# Divulgação de informação sensível: a citação que veio de outra faculdade

OWASP Top 10 para LLMs 2025 — **LLM02 Sensitive Information Disclosure**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
O Acervo Aurora mantém um único índice com documentos de programas diferentes (PPG-Computação e PPG-Saúde). Uma aluna do PPG-Computação faz uma pergunta e a busca por semelhança não verifica a que programa ela pertence.

In [2]:
ALUNO = "A-1001"  # Marina Tavares, PPG-Computação
PERGUNTA = "Existe algum estudo sobre adesão a protocolos em laboratórios?"
programa_aluno = dados.ALUNOS[ALUNO]["programa"]
print(ALUNO, "->", programa_aluno)


A-1001 -> PPG-Computação


## Sem defesa
O índice é construído com TODO o acervo, sem filtrar por programa.

In [3]:
indice_vulneravel = dados.Indice(dados.ACERVO)
achados = indice_vulneravel.buscar(PERGUNTA, k=1)
for d in achados:
    print(d["id"], "-", d["programa"], "-", d["titulo"])
assert achados[0]["programa"] == "PPG-Saúde"  # a aluna é do PPG-Computação


T-S-004 - PPG-Saúde - Adesão a protocolos de higienização em laboratórios


## Com defesa
`defesas.filtrar_por_programa` remove, ANTES da busca, qualquer documento que não seja do programa da aluna (ou público).

In [4]:
acervo_filtrado = defesas.filtrar_por_programa(dados.ACERVO, programa_aluno)
indice_defendido = dados.Indice(acervo_filtrado)
achados_defendidos = indice_defendido.buscar(PERGUNTA, k=1)
for d in achados_defendidos:
    print(d["id"], "-", d["programa"], "-", d["titulo"])
assert all(d["programa"] in (programa_aluno, "Público") for d in achados_defendidos)


## O que observar
- A busca por semelhança mede proximidade de texto, não permissão: o documento mais parecido com a pergunta venceu, não importa de quem é.
- O vazamento aqui não depende de nenhuma instrução maliciosa — é um problema de arquitetura: um índice só, sem isolamento por tenant/programa.
- Filtrar por programa ANTES da busca (e não revisar a resposta depois) é o que realmente impede o documento de aparecer.